# Chapter 5: Seakeeping Models — Wave Spectrum, RAOs, and Fluid Resonance

In Chapter 4, we treated the ocean as completely flat and calm, using hydrostatics to find our vessel's natural oscillation period ($T_\phi$). In **Chapter 5**, we introduce the external driving force: **Ocean Waves**.

Seakeeping is the study of how a marine vehicle responds to a dynamic, irregular seaway. Instead of tracking individual waves, we use statistical and fluid-dynamic models to predict how much the hull will heave, pitch, and roll. In Fossen's framework, this adds a time-varying excitation force vector $\tau_{\text{wave}}(t)$ to our equations of motion:

$$M \dot{\nu} + C(\nu)\nu + D(\nu)\nu + g(\eta) = \tau_{\text{propulsion}} + \tau_{\text{wave}}(t)$$

To build a GNC system that can anticipate wave slamming or avoid parametric resonance, we must master two core concepts:
1. **Wave Spectra:** How we mathematically represent the random, chaotic energy of the ocean surface.
2. **Response Amplitude Operators (RAOs):** The transfer functions that define exactly how much our specific hull translates a wave of frequency $\omega$ into physical vehicle motion.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def modified_pierson_moskowitz(omega, h_s, t_z):
    """
    Computes the wave energy density spectrum using the Modified Pierson-Moskowitz 
    (or Bretschneider) formulation for a given significant wave height and zero-crossing period.
    """
    # Prevent division by zero for the omega array
    omega = np.maximum(omega, 0.001)
    
    A = 4.0 * (np.pi**3) * (h_s**2) / (t_z**4)
    B = 16.0 * (np.pi**3) / (t_z**4)
    
    # S(\omega) = A / \omega^5 * exp(-B / \omega^4)
    spectrum = (A / (omega**5)) * np.exp(-B / (omega**4))
    return spectrum

# Test generation array for wave frequencies (rad/s)
omega_vals = np.linspace(0.1, 2.5, 200)
print("Wave Spectrum Function Loaded Successfully.")

Wave Spectrum Function Loaded Successfully.


---

## 1. Modeling Irregular Seas: The Wave Spectrum

The ocean surface is a chaotic superposition of thousands of individual sine waves generated by wind blowing over vast stretches of water (the fetch). To model this computationally, we use a power spectral density function. 

The **Modified Pierson-Moskowitz spectrum** models a fully developed sea using two primary parameters:
* **Significant Wave Height ($H_s$):** The average height (crest to trough) of the highest one-third of all waves.
* **Zero-Crossing Period ($T_z$):** The average time interval between successive upward crossings of the mean water level.

Let's look at how the energy profile of the ocean shifts as a storm brews. Notice how as the wind blows harder, the peak of the wave energy not only grows taller, but shifts to a **lower frequency** (longer wave periods).

In [2]:
def plot_wave_spectrum(sig_wave_height, zero_cross_period):
    """
    Plots the Pierson-Moskowitz wave energy spectrum against wave frequency.
    """
    omega = np.linspace(0.1, 2.5, 300)
    energy_density = modified_pierson_moskowitz(omega, sig_wave_height, zero_cross_period)
    
    # Find peak frequency for annotation
    peak_idx = np.argmax(energy_density)
    peak_omega = omega[peak_idx]
    peak_period = (2.0 * np.pi) / peak_omega

    plt.figure(figsize=(10, 4.5))
    plt.plot(omega, energy_density, color='teal', linewidth=3, label=r'Wave Energy Spectrum $S(\omega)$')
    plt.fill_between(omega, energy_density, color='teal', alpha=0.15)
    
    # Structural Annotations
    plt.axvline(peak_omega, color='darkorange', linestyle='--', alpha=0.8, 
                label=f'Peak Energy Freq ({peak_omega:.2f} rad/s / T={peak_period:.1f}s)')
    
    plt.title(f"Ocean Sea State Spectrum | Significant Wave Height ($H_s$): {sig_wave_height}m", fontsize=12)
    plt.xlabel(r"Wave Encounter Frequency ($\omega$) [rad/s]", fontsize=10)
    plt.ylabel(r"Energy Density [m² \cdot s]", fontsize=10)
    plt.xlim(0.1, 2.5)
    plt.ylim(0, max(1.0, np.max(energy_density) * 1.2))
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right')
    plt.show()

interact(plot_wave_spectrum,
         sig_wave_height=widgets.FloatSlider(min=0.5, max=8.0, step=0.5, value=3.0, description='H_s (Waves m):'),
         zero_cross_period=widgets.FloatSlider(min=4.0, max=14.0, step=0.5, value=8.0, description='T_z (Period s):'));

interactive(children=(FloatSlider(value=3.0, description='H_s (Waves m):', max=8.0, min=0.5, step=0.5), FloatS…

---

## 2. Response Amplitude Operators (RAOs)

A wave spectrum tells us what the *ocean* is doing, but it doesn't tell us what the *vessel* is doing. To bridge that gap, we use a **Response Amplitude Operator (RAO)**. 

An RAO is effectively a fluid-dynamic transfer function $H(\omega)$ in the frequency domain. It maps input wave amplitude to output vessel motion amplitude:

$$\text{RAO}(\omega) = \frac{\text{Vessel Motion Amplitude (e.g., Roll Degrees)}}{\text{Wave Amplitude (Meters)}}$$



The shape of the RAO is entirely determined by the vehicle's physical properties: its dry mass, its hydrodynamic added mass, its damping, and its hydrostatic righting arm ($GM$). The peak of the RAO corresponds precisely to the vessel's **natural frequency** ($\omega_0$) that we explored in Chapter 4.

In [10]:
def plot_seakeeping_response(vessel_gm, wave_hz_period):
    """
    Combines the fixed ocean wave spectrum with the vessel's RAO to show 
    the resulting response spectrum. Demonstrates pure wave-body resonance.
    """
    omega = np.linspace(0.1, 2.5, 300)
    
    # 1. Generate the Sea State (Driving Force)
    h_s = 3.5  # Fixed 3.5m storm waves
    t_z = wave_hz_period  # Dynamic ocean period from slider
    sea_spectrum = modified_pierson_moskowitz(omega, h_s, t_z)
    
    # 2. Derive the Vessel's RAO (The Transfer Function)
    # Under the hood, the vessel's natural frequency is tuned by the GM slider
    m = 1200000.0
    g = 9.81
    ix_total = (1.0 / 12.0) * m * (12.0**2) * 1.4  # Includes added mass
    omega_natural = np.sqrt((m * g * vessel_gm) / ix_total)
    
    # Linear mass-spring-damper RAO model approximation for Roll response
    damping_ratio = 0.12  # Fluid damping factor
    rao = (omega_natural**2) / np.sqrt((omega_natural**2 - omega**2)**2 + (2 * damping_ratio * omega_natural * omega)**2)
    
    # 3. Compute the Response Spectrum: S_response(\omega) = S_sea(\omega) * |RAO(\omega)|^2
    response_spectrum = sea_spectrum * (rao**2)
    
    # Plotting layout
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
    
    # Left Plot: The Sea State vs The Vessel's Mechanical Susceptibility
    ax1.plot(omega, sea_spectrum / np.max(sea_spectrum), color='teal', linewidth=2, label='Ocean Wave Energy (Normalized)')
    ax1.plot(omega, rao / np.max(rao), color='darkviolet', linewidth=2.5, label='Vessel Roll RAO (Sensitivity)')
    ax1.set_title("Input Sea Spectrum vs. Structural RAO Profile", fontsize=11)
    ax1.set_xlabel(r"Encounter Frequency ($\omega$) [rad/s]")
    ax1.set_ylabel("Normalized Magnitude")
    ax1.grid(True, linestyle=':')
    ax1.legend(loc='upper right')
    
    # Right Plot: The Reality (The Co-Multiplied Spectrum)
    ax2.plot(omega, response_spectrum, color='crimson', linewidth=3, label='Resulting Roll Response Spectrum')
    ax2.fill_between(omega, response_spectrum, color='crimson', alpha=0.15)
    ax2.set_title("Actual Vessel Roll Energy Spectrum", fontsize=11)
    ax2.set_xlabel(r"Encounter Frequency ($\omega$) [rad/s]")
    ax2.set_ylabel(r"Motion Power Density [deg² \cdot s]")
    ax2.grid(True, linestyle=':')
    
    # Detect Resonance overlap
    peak_sea_omega = omega[np.argmax(sea_spectrum)]
    if abs(peak_sea_omega - omega_natural) < 0.15:
        rect_color = 'salmon'
        # Fixed: Changed emoji to [!] and added 'r' prefix for safety
        status_msg = r"[!] CRITICAL RESONANCE INJECTION:\nWave energy lines up perfectly\nwith hull dynamics! Extreme rolling."
    else:
        rect_color = 'honeydew'
        # Fixed: Changed emoji to [OK] and added 'r' prefix for safety
        status_msg = r"[OK] SAFE SEAKEEPING:\nEnergy profiles decoupled.\nHull dampens out wave energy."
        
    ax2.text(1.2, np.max(response_spectrum)*0.7 if np.max(response_spectrum) > 0 else 1, status_msg,
             bbox=dict(boxstyle='round', facecolor=rect_color, alpha=0.8), family='monospace', fontsize=9)
    
    plt.tight_layout()
    plt.show()

interact(plot_seakeeping_response,
         vessel_gm=widgets.FloatSlider(min=0.2, max=2.2, step=0.1, value=0.8, description='Vessel GM (m):'),
         wave_hz_period=widgets.FloatSlider(min=5.0, max=13.0, step=0.5, value=11.0, description='Ocean Wave T_z(s):'));

interactive(children=(FloatSlider(value=0.8, description='Vessel GM (m):', max=2.2, min=0.2), FloatSlider(valu…

---

## 3. Fluid Memory Effects & The Retardation Function ($K(t)$)

When a submarine or ship moves in waves, it radiates fluid energy away from the hull. This creates a "memory effect" in the water: the hydrodynamic force acting on the hull *right now* depends on what the vehicle was doing *in the past*. 

To capture this, we drop constant added mass and damping terms and implement **Cummins’ Equation**:

$$[M + A(\infty)]\dot{\nu}(t) + \int_{0}^{t} K(t-\tau)\nu(\tau)d\tau + g(\eta) = \tau_{\text{wave}}(t)$$

Where $A(\infty)$ is the instantaneous added mass at infinite frequency, and $K(t)$ is the **Retardation Function (Memory Kernel)**. Because a running convolution integral ($\int K \cdot \nu$) is computationally too expensive for a real-time GNC flight controller, we approximate $K(t)$ in the time domain using a **State-Space Identification** model:

$$\dot{x}_r(t) = A_r x_r(t) + B_r \nu(t)$$
$$\tau_{\text{radiation}}(t) \approx C_r x_r(t)$$

Let's simulate how this fluid memory behaves compared to a simple, memoryless damper after a hull receives an impact.

In [11]:
def simulate_fluid_memory(memory_duration_seconds):
    """
    Simulates a 4th-order state-space approximation of the fluid retardation function K(t).
    Compares a system with true fluid memory decay against a standard constant-damping model.
    """
    t = np.linspace(0, 25, 250)
    dt = t[1] - t[0]
    
    # State-space realization matrices for a typical mid-sized hull's roll fluid memory
    # This acts as a low-pass filter tracking the history of the roll velocity
    A_r = np.array([
        [-0.5, -1.2,  0.0,  0.0],
        [ 1.0,  0.0,  0.0,  0.0],
        [ 0.0,  1.0,  0.0, -0.8],
        [ 0.0,  0.0,  1.0, -0.1]
    ])
    B_r = np.array([[1.0], [0.0], [0.0], [0.0]])
    C_r = np.array([[0.0, 0.4, 0.1, 0.05]]) * memory_duration_seconds
    
    # Initialize state variables
    x_r = np.zeros((4, 1))
    roll_angle_mem = np.zeros_like(t)
    roll_vel_mem = np.zeros_like(t)
    
    roll_angle_const = np.zeros_like(t)
    roll_vel_const = np.zeros_like(t)
    
    # Initial conditions: Kick the boat to a 15-degree roll angle
    roll_angle_mem[0] = 15.0
    roll_angle_const[0] = 15.0
    
    # System physical constants (Stiffness and Inertia)
    omega_n = 0.7  # Structural natural frequency
    constant_damping_factor = 0.15
    
    # Time-stepping simulation loop
    for i in range(1, len(t)):
        # 1. System with Fluid Memory
        mem_force = (C_r @ x_r).item()
        acc_mem = - (omega_n**2) * roll_angle_mem[i-1] - 0.05 * roll_vel_mem[i-1] - mem_force
        roll_vel_mem[i] = roll_vel_mem[i-1] + acc_mem * dt
        roll_angle_mem[i] = roll_angle_mem[i-1] + roll_vel_mem[i-1] * dt
        
        # Update fluid memory states via Euler integration
        dx_r = A_r @ x_r + B_r * roll_vel_mem[i-1]
        x_r += dx_r * dt
        
        # 2. Benchmark System with Constant Damping (No Memory)
        acc_const = - (omega_n**2) * roll_angle_const[i-1] - constant_damping_factor * roll_vel_const[i-1]
        roll_vel_const[i] = roll_vel_const[i-1] + acc_const * dt
        roll_angle_const[i] = roll_angle_const[i-1] + roll_vel_const[i-1] * dt

    plt.figure(figsize=(11, 4.5))
    plt.plot(t, roll_angle_const, 'k--', alpha=0.6, label='Constant Damping (Chapter 4 Linear)')
    plt.plot(t, roll_angle_mem, color='darkviolet', linewidth=2.5, label="Cummins' Fluid Memory Model")
    
    plt.title("Fluid Radiation Impact: Constant Damping vs. Dynamic Memory Kernel", fontsize=12)
    plt.xlabel("Time Elapsed (Seconds)", fontsize=10)
    plt.ylabel("Roll Angle (Degrees)", fontsize=10)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(loc='upper right')
    plt.show()

interact(simulate_fluid_memory,
         memory_duration_seconds=widgets.FloatSlider(min=0.0, max=3.0, step=0.2, value=1.4, description='Memory Intensity:'));

interactive(children=(FloatSlider(value=1.4, description='Memory Intensity:', max=3.0, step=0.2), Output()), _…

---

## 4. Time-Domain Realization (Longcrested Irregular Seas)

To feed a true time-varying excitation force $\tau_{\text{wave}}(t)$ into our Guidance and Control models, we must transform the frequency-domain spectrum $S(\omega)$ into a continuous time-series wave profile. 

Using the **Principle of Superposition**, we discretize the Pierson-Moskowitz spectrum into $N$ distinct frequency components. We extract the exact energy amplitude for each slice, assign a completely random phase angle ($\epsilon_i \in [0, 2\pi]$), and sum them up as a series of harmonically moving waves:

$$\zeta(t) = \sum_{i=1}^{N} \sqrt{2 S(\omega_i) \Delta\omega} \cos(\omega_i t + \epsilon_i)$$

Let's generate a living, irregular sea state time realization from our underlying spectrum. This represents the actual, chaotic water elevation our sub encounters at sea.

In [14]:
def generate_time_domain_waves(h_s, t_z, seed_value):
    """
    Discretizes the wave spectrum and uses superposition to generate
    a true, chaotic time-domain wave elevation profile.
    """
    np.random.seed(seed_value)
    
    # Component configuration
    N = 60  # Number of wave frequency components to superimpose
    omega = np.linspace(0.1, 2.2, N)
    d_omega = omega[1] - omega[0]
    
    # Calculate energy spectrum amplitudes
    S_w = modified_pierson_moskowitz(omega, h_s, t_z)
    wave_amplitudes = np.sqrt(2.0 * S_w * d_omega)
    
    # Assign random phase angles to each component
    phases = np.random.uniform(0, 2.0 * np.pi, N)
    
    # Time execution array
    t = np.linspace(0, 60, 500)
    wave_elevation = np.zeros_like(t)
    
    # Superimpose individual wave components
    for i in range(N):
        wave_elevation += wave_amplitudes[i] * np.cos(omega[i] * t + phases[i])
        
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6.5), gridspec_kw={'height_ratios': [1, 1.5]})
    
    # Plot 1: Underlying Spectrum
    ax1.plot(omega, S_w, color='teal', linewidth=2)
    ax1.fill_between(omega, S_w, color='teal', alpha=0.1)
    ax1.set_title(r"Target Ocean Spectrum $S(\omega)$", fontsize=10)
    ax1.set_ylabel("Energy")
    ax1.grid(True, linestyle=':')
    
    # Plot 2: Chaotic Time Realization
    ax2.plot(t, wave_elevation, color='dodgerblue', linewidth=2.5, label='Irregular Sea Elevation')
    ax2.axhline(0, color='black', linestyle='--', alpha=0.4)
    
    # Annotate significant height lines
    ax2.axhline(h_s/2, color='crimson', linestyle=':', alpha=0.7, label='Significant Wave Height Limit Threshold')
    ax2.axhline(-h_s/2, color='crimson', linestyle=':', alpha=0.7)
    
    ax2.set_title(r"Real-Time Ocean Surface Elevation $\zeta(t)$ | Encounter Window", fontsize=11)
    ax2.set_xlabel("Time (Seconds)", fontsize=10)
    ax2.set_ylabel("Wave Elevation (Meters)", fontsize=10)
    ax2.grid(True, linestyle=':')
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

interact(generate_time_domain_waves,
         h_s=widgets.FloatSlider(min=1.0, max=7.0, step=0.5, value=3.5, description='H_s (Height m):'),
         t_z=widgets.FloatSlider(min=5.0, max=13.0, step=0.5, value=9.0, description='T_z (Period s):'),
         seed_value=widgets.IntSlider(min=1, max=100, step=1, value=42, description='Storm Seed:'));

interactive(children=(FloatSlider(value=3.5, description='H_s (Height m):', max=7.0, min=1.0, step=0.5), Float…

---

## Summary of Chapter 5 Seakeeping Principles

1. **Ocean Waves** are irregular and must be handled statistically in the frequency domain using energy density charts like the **Pierson-Moskowitz Spectrum**.
2. **The Response Amplitude Operator (RAO)** is the vehicle’s dynamic fingerprint. It defines how vulnerable the hull is to different wave frequencies.
3. **The Response Spectrum** is the final mathematical product of the Sea Spectrum and the square of the RAO. 
4. **GNC Mitigation:** If your software detects that the ocean's peak energy frequency is creeping toward the peak of the RAO sensitivity curve, it must trigger a mitigation matrix: either changing headings to alter the *encounter frequency* (Doppler effect) or adjusting ballast tanks to warp the $GM$ and shift the RAO peak entirely out of harm's way.